In [0]:
-- Create a temporary view of your table if not already
CREATE OR REPLACE TEMP VIEW loan_view AS
SELECT *
FROM big_query_catalog.credit_risk_main.main_data;

In [0]:
SELECT * FROM loan_view

LIMIT 50;

In [0]:
CREATE OR REPLACE TEMP VIEW loans_preprocessed AS
SELECT *,
       to_date(issue_d, 'M/d/yyyy')            AS issue_date_parsed,
       to_date(earliest_cr_line, 'M/d/yyyy')   AS earliest_cr_line_parsed,
       to_date(last_pymnt_d, 'M/d/yyyy')       AS lastpymt_date_parsed,
       to_date(last_credit_pull_d, 'M/d/yyyy') AS lastcreditpull_date_parsed,
       to_date(next_pymnt_d, 'M/d/yyyy')       AS next_payment_date_parsed
FROM loan_view;


In [0]:
select * from loans_preprocessed

limit 5;

In [0]:
CREATE OR REPLACE TEMP VIEW loans_clean AS
SELECT
    -- Replace issue_date with the parsed value
    issue_date_parsed AS issue_date,
    
    -- Keep all other columns except the old issue_date & issue_date_parsed
    *
EXCEPT(issue_date, issue_date_parsed)
FROM loans_preprocessed;


In [0]:
select * from loans_clean

limit 5;

In [0]:
SHOW CATALOGS;

In [0]:
show schemas in system;

In [0]:
--  Creating a catalog, Schema for Processed Data

CREATE CATALOG data_processed;


In [0]:
CREATE SCHEMA data_processed.credit_risk_main;


In [0]:
-- Create a new Temp view for further data pre-processing

CREATE OR REPLACE TEMP VIEW loan_cleaned AS
SELECT *
FROM data_processed.credit_risk_main.loans_preprocessed;

In [0]:
-- Summarizing Categorical Features

SELECT
    home_ownership,
    COUNT(*) AS count
FROM loan_cleaned
GROUP BY home_ownership
ORDER BY count DESC;


In [0]:
SELECT
    purpose,
    COUNT(*) AS count
FROM loan_cleaned
GROUP BY purpose
ORDER BY count DESC;

In [0]:
SELECT
    title,
    COUNT(*) AS count
FROM loan_cleaned
GROUP BY title
ORDER BY count DESC;

In [0]:
-- Finding NUll Values for efficient Handling

SELECT
  COUNT(CASE WHEN acc_now_delinq IS NULL THEN 1 END) AS null_values
FROM loan_cleaned;

In [0]:
-- Feature Engineering and Handling Missing Values

CREATE OR REPLACE VIEW loan_cleaned_final AS
SELECT
    id,
    member_id,
    issue_date,
    earliest_cr_line_parsed,
    lastpymt_date_parsed,
    lastcreditpull_date_parsed,
    next_payment_date_parsed,
    
    COALESCE(loan_amnt, 0) AS loan_amnt,
    COALESCE(funded_amnt, 0) AS funded_amnt,
    COALESCE(funded_amnt_inv, 0) AS funded_amnt_inv,

    -- Extract numeric months from term
    COALESCE(TRY_CAST(regexp_extract(term, '(\\d+)', 1) AS INT), 0) AS term,

    COALESCE(int_rate, 0) AS int_rate,
    COALESCE(installment, 0) AS installment,
    COALESCE(grade, 'Not Available') AS grade,
    COALESCE(sub_grade, 'Not Available') AS sub_grade,

    -- Clean emp_length
    COALESCE(
        CASE 
            WHEN emp_length IS NULL OR emp_length = '' THEN 0
            WHEN emp_length LIKE '< 1%' THEN 1
            ELSE TRY_CAST(regexp_extract(emp_length, '(\\d+)', 1) AS INT)
        END,
    0) AS emp_length_num,

    COALESCE(emp_title, 'Not Available') AS emp_title,
    COALESCE(home_ownership, 'Not Available') AS home_ownership,
    COALESCE(annual_inc, 0) AS annual_inc,
    COALESCE(verification_status, 'Not Available') AS verification_status,
    
    COALESCE(pymnt_plan, 'Unknown') AS pymnt_plan,
    COALESCE(desc, 'Not Available') AS desc,
    COALESCE(purpose, 'Not Available') AS purpose,
    COALESCE(title, 'Not Available') AS title,
    COALESCE(zip_code, 'Not Available') AS zip_code,
    COALESCE(addr_state, 'Not Available') AS addr_state,

    COALESCE(dti, 0) AS dti,
    COALESCE(delinq_2yrs, 0) AS delinq_2yrs,
    
    COALESCE(inq_last_6mths, 0) AS inq_last_6mths,
    COALESCE(mths_since_last_delinq, 0) AS mths_since_last_delinq,
    COALESCE(mths_since_last_record, 0) AS mths_since_last_record,
    COALESCE(open_acc, 0) AS open_acc,
    COALESCE(pub_rec, 0) AS pub_rec,
    COALESCE(revol_bal, 0) AS revol_bal,
    COALESCE(revol_util, 0) AS revol_util,
    COALESCE(total_acc, 0) AS total_acc,
    COALESCE(initial_list_status, 'Not Available') AS initial_list_status,

    COALESCE(out_prncp, 0) AS out_prncp,
    COALESCE(out_prncp_inv, 0) AS out_prncp_inv,
    COALESCE(total_pymnt, 0) AS total_pymnt,
    COALESCE(total_pymnt_inv, 0) AS total_pymnt_inv,
    COALESCE(total_rec_prncp, 0) AS total_rec_prncp,
    COALESCE(total_rec_int, 0) AS total_rec_int,
    COALESCE(total_rec_late_fee, 0) AS total_rec_late_fee,
    COALESCE(recoveries, 0) AS recoveries,
    COALESCE(collection_recovery_fee, 0) AS collection_recovery_fee,

    COALESCE(last_pymnt_amnt, 0) AS last_pymnt_amnt,
    
    COALESCE(collections_12_mths_ex_med, 0) AS collections_12_mths_ex_med,
    COALESCE(mths_since_last_major_derog, 0) AS mths_since_last_major_derog,
    COALESCE(policy_code, 0) AS policy_code,
    COALESCE(application_type, 'Not Available') AS application_type,
    COALESCE(annual_inc_joint, 0) AS annual_inc_joint,
    COALESCE(dti_joint, 0) AS dti_joint,
    COALESCE(verification_status_joint, 'Not Available') AS verification_status_joint,

    COALESCE(acc_now_delinq, 0) AS acc_now_delinq,
    COALESCE(tot_coll_amt, 0) AS tot_coll_amt,
    COALESCE(tot_cur_bal, 0) AS tot_cur_bal,
    COALESCE(open_acc_6m, 0) AS open_acc_6m,
    COALESCE(open_il_6m, 0) AS open_il_6m,
    COALESCE(open_il_12m, 0) AS open_il_12m,
    COALESCE(open_il_24m, 0) AS open_il_24m,
    COALESCE(mths_since_rcnt_il, 0) AS mths_since_rcnt_il,
    COALESCE(total_bal_il, 0) AS total_bal_il,
    COALESCE(il_util, 0) AS il_util,
    COALESCE(open_rv_12m, 0) AS open_rv_12m,
    COALESCE(open_rv_24m, 0) AS open_rv_24m,
    COALESCE(max_bal_bc, 0) AS max_bal_bc,
    COALESCE(all_util, 0) AS all_util,
    COALESCE(total_rev_hi_lim, 0) AS total_rev_hi_lim,
    COALESCE(inq_fi, 0) AS inq_fi,
    COALESCE(total_cu_tl, 0) AS total_cu_tl,
    COALESCE(inq_last_12m, 0) AS inq_last_12m,

    CASE 
        WHEN next_payment_date_parsed IS NULL THEN 'No further payment'
        ELSE 'Pending payment'
    END AS payment_status

   FROM data_processed.credit_risk_main.loans_preprocessed;


In [0]:
SELECT * FROM 

loan_cleaned_final

limit 20;

In [0]:
CREATE OR REPLACE TABLE data_processed.credit_risk_main.loans_transformed AS
SELECT
    *
FROM loan_cleaned_final;